# 环节 05 · 资源限制与生命周期演示

纯 Python 标准库，零依赖。手搓四件事：

1. cgroup v2 配额判定（CPU 限流 / 内存 OOM / pids 触顶）；
2. wall-clock 与软超时的组合；
3. 预热池排队与冷启动摊薄；
4. 单机密度估算。

In [ ]:
# §1 cgroup 配额判定
def cpu_check(usage_cores, cpu_max):
    quota = cpu_max[0] / cpu_max[1]      # cpu.max = "$MAX $PERIOD"
    if usage_cores > quota:
        return f"THROTTLED（限流到 {quota:.2f} 核）"
    return "OK"


def mem_check(usage_mb, memory_max_mb):
    if usage_mb > memory_max_mb:
        return "OOM_KILL（进程被杀）"
    return "OK"


def pids_check(procs, pids_max):
    if procs >= pids_max:
        return "EAGAIN（fork 失败）"
    return "OK"


print("CPU :", cpu_check(usage_cores=1.4, cpu_max=(50000, 100000)))
print("内存:", mem_check(usage_mb=600, memory_max_mb=512))
print("进程:", pids_check(procs=256, pids_max=256))
print()
print("记忆点：cpu.max 只『限流不杀』；memory.max 触顶是 OOM kill；pids 触顶是 EAGAIN")

In [ ]:
# §2 超时组合：只设一个都不够
def supervise(elapsed_s, idle_s, cpu_s, wall_limit, idle_limit, cpu_limit):
    events = []
    if elapsed_s >= wall_limit:
        events.append("WALL-CLOCK 超时 → SIGKILL（兜底，必须有）")
    if idle_s >= idle_limit:
        events.append("无输出软超时 → 告警/终止（交互场景）")
    if cpu_s >= cpu_limit:
        events.append("CPU 配额触顶 → 限流（不终止）")
    return events or ["运行中"]


print("正常任务  :", supervise(5, 0, 3, wall_limit=60, idle_limit=30, cpu_limit=60))
print("sleep 攻击:", supervise(45, 45, 0, wall_limit=60, idle_limit=30, cpu_limit=60))
print("死循环    :", supervise(10, 0, 60, wall_limit=60, idle_limit=30, cpu_limit=60))
print()
print("补充：还要对 stdout/stderr 做滚动截断（防 echo 无限刷爆缓冲）")

In [ ]:
# §3 预热池：冷启动怎么被摊薄
def experience(task_ms, cold_ms, pool_hit_rate):
    warm = 0
    cold = cold_ms
    return (pool_hit_rate * (warm + task_ms)
            + (1 - pool_hit_rate) * (cold + task_ms))


TASK = 300
COLD = 2100
for hit in [0.0, 0.5, 0.9, 0.99]:
    avg = experience(TASK, COLD, hit)
    print(f"预热命中率 {hit:>5.0%} → 平均体验 {avg:>7.0f} ms"
          f"（无池时 {COLD + TASK} ms）")
print()
print("代价：池空转成本 + 『领到即干净』的复位要求（否则跨任务数据泄露）")

In [ ]:
# §4 单机密度估算（内存 + CPU + pids 三约束取最小）
def estimate(host_mem_gb=48, host_cores=12, host_pids=20000,
             fixed_mb=10, task_mem_mb=300, task_cores=0.5, task_pids=40,
             reserve=0.2):
    mem_avail = host_mem_gb * 1024 * (1 - reserve)
    by_mem = int(mem_avail / (fixed_mb + task_mem_mb))
    by_cpu = int(host_cores * (1 - reserve) / task_cores)
    by_pids = int(host_pids * (1 - reserve) / task_pids)
    return min(by_mem, by_cpu, by_pids), by_mem, by_cpu, by_pids


n, m, c, p = estimate()
print(f"单机上限 = {n}（内存 {m} / CPU {c} / pids {p}）")
print()
print("注意：① 用 P99 峰值而不是均值；② 留 20% 余量；")
print("      ③ 别忘了输出缓冲与临时文件（见环节 03 的 copy-up）")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | 只设超时够不够？ | 不够；30 秒内能吃完内存/fork 炸弹 |
| 2 | `cpu.max` 触顶会怎样？ | 限流变慢，不杀进程 |
| 3 | `memory.max` 触顶的表现？ | OOM kill（进程突然消失） |
| 4 | wall-clock 与 CPU time 为什么分开？ | 前者挡 sleep/等网络，后者只反映算力 |
| 5 | 预热池最大风险？ | 状态不复位 → 跨任务/跨租户数据泄露 |
| 6 | 冷启动优化先做哪一步？ | 先量应用初始化占比，再决定快照/预热 |

**相关长文**：[环节05-资源限制与生命周期详解.md](./环节05-资源限制与生命周期详解.md)